# Planners-11: Unified Planning

**Navigation** : [Index](../../README.md) | [<< LLM Planning](Planners-10-LLM-Planning.ipynb) | [LOOP >>](Planners-12-LOOP.ipynb)

## Interface Unifiee pour Multi-Solveurs

Ce notebook presente **unified-planning**, une bibliotheque Python qui offre une interface unique pour interagir avec de multiples planificateurs (Fast Downward, Pyperplan, Tamer, etc.). Cette abstraction permet de comparer facilement differents solveurs et de changer de planificateur sans reecrire le code.

### Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. Utiliser l'API unified-planning pour definir des problemes de planification
2. Comparer les performances de differents solveurs sur un meme probleme
3. Exploiter les fonctionnalites avancees : planification temporelle, fluents numeriques
4. Convertir entre le format Python et PDDL standard
5. Choisir le solveur adapte selon le type de probleme
6. Integrer unified-planning dans un pipeline de planification

### Prerequis
- Avoir suivi [Planners-2-PDDL-Basics](../01-Foundation/Planners-2-PDDL-Basics.ipynb)
- Connaissance de Python oriente objet
- Comprendre le modele STRIPS (predicats, actions, preconditions, effets)

### Duree estimee : 40 minutes

---

## 1. Introduction a unified-planning

**unified-planning** est un projet de l'Union Europeenne (AIPlan4EU) qui vise a democratise l'acces aux outils de planification. Elle offre :

- Une API Python intuitive pour definir domaines et problemes
- Une interface unifiee pour 10+ planificateurs
- Support pour PDDL, planification temporelle, et contraintes numeriques
- Comparaison automatique des performances

### 1.1 Planificateurs supportes

| Planificateur | Type | Optimalite | Specialites |
|---------------|------|------------|--------------|
| **Fast Downward** | Classique | Optimal/Satisficing | Heuristiques LM-cut, FF |
| **Pyperplan** | Classique | Satisficing | Leger, rapide |
| **Tamer** | Classique | Satisficing | Planification avec preferences |
| **ENHSP** | Numerique | Optimal | Fluents numeriques, temps |
| **UP-SMT** | SMT | Optimal | Contraintes complexes |
| **LPG** | Temporel | Satisficing | PDDL 2.1, durees |
| **OSP** | Oversubscription | Partiel | Objectifs multiples |

### 1.2 Installation et configuration

La bibliotheque s'installe avec pip. Chaque planificateur a son propre package optionnel.

In [1]:
# Installation des packages necessaires
# Decommentez les lignes si necessaire

# !pip install unified-planning
# !pip install up-fast-downward  # Fast Downward
# !pip install up-pyperplan      # Pyperplan
# !pip install up-tamer          # Tamer

# Imports principaux
import sys
import time
from typing import List, Dict, Optional, Any

from unified_planning.shortcuts import *
# Note: En unified-planning 1.3.0+, toutes les classes sont importees via shortcuts
# UserType, BoolType, IntType viennent de shortcuts, pas de model.types
from unified_planning.engines import Engine
from unified_planning.model import Problem, Action, Fluent, Object

# Imports pour les resultats
from unified_planning.engines.results import PlanGenerationResultStatus

print("unified-planning version:", __import__('unified_planning').__version__)

unified-planning version: 1.3.0


Verification des moteurs de planification disponibles sur le systeme et affichage de leurs capacites (planification classique, temporelle, numerique, etc.).

In [2]:
# Verification des moteurs disponibles
def check_available_engines():
    """Liste les planificateurs disponibles sur le systeme."""
    from unified_planning.environment import get_environment
    from unified_planning.engines import Factory
    
    # unified-planning 1.3.0+ requiere un environnement
    env = get_environment()
    factory = Factory(env)
    
    # factory.engines retourne une liste de noms de moteurs
    engines_list = factory.engines
    
    print("Planificateurs disponibles:")
    print("=" * 50)
    
    for name in sorted(engines_list):
        # Verifier si c'est un planificateur
        if 'oneshot' in name.lower() or 'planner' in name.lower() or name in ['pyperplan', 'fast-downward', 'tamer']:
            print(f"  - {name}")
    
    return engines_list

available = check_available_engines()
print(f"\nTotal: {len(available)} moteurs detectes")

Planificateurs disponibles:
  - fast-downward
  - replanner[fast-downward-opt]
  - replanner[fast-downward]

Total: 21 moteurs detectes


---

## 2. Definition de Problemes avec l'API

L'API unified-planning permet de definir des problemes directement en Python, sans ecrire de fichiers PDDL. C'est souvent plus intuitif et permet l'integration avec d'autres outils Python.

### 2.1 Types et Fluents

Les **types utilisateur** (UserType) representent les objets du domaine. Les **fluents** sont des fonctions qui retournent une valeur selon l'etat du monde.

In [3]:
# Creation d'un probleme de navigation robot
# Le robot doit se deplacer d'un point A a un point B

# Definition des types
Location = UserType('Location')
Robot = UserType('Robot')

# Definition des fluents (predicats variables)
# robot_at(r, l) = le robot r est a la location l
robot_at = Fluent('robot_at', BoolType(), r=Robot, l=Location)

# connected(l1, l2) = la location l1 est connectee a l2 (graphe statique)
connected = Fluent('connected', BoolType(), l1=Location, l2=Location)

# visited(l) = la location l a ete visitee
visited = Fluent('visited', BoolType(), l=Location)

print("Types et Fluents definis:")
print(f"  Types: Location, Robot")
print(f"  Fluents: robot_at, connected, visited")

Types et Fluents definis:
  Types: Location, Robot
  Fluents: robot_at, connected, visited


### 2.2 Actions et Effets

Une **action** a des parametres, des preconditions et des effets. L'API utilise des methodes fluentes pour ajouter ces elements.

In [4]:
# Definition de l'action 'move'
# Le robot se deplace d'une location a une autre connectee

move = InstantaneousAction('move', r=Robot, l_from=Location, l_to=Location)
r = move.parameter('r')
l_from = move.parameter('l_from')
l_to = move.parameter('l_to')

# Preconditions
move.add_precondition(connected(l_from, l_to))  # Les lieux sont connectes
move.add_precondition(robot_at(r, l_from))      # Le robot est au depart

# Effets
move.add_effect(robot_at(r, l_from), False)     # Le robot quitte l_from
move.add_effect(robot_at(r, l_to), True)        # Le robot arrive a l_to
move.add_effect(visited(l_to), True)            # Marquer l_to comme visite

print("Action 'move' definie:")
print(f"  Parametres: r:Robot, l_from:Location, l_to:Location")
print(f"  Preconditions: connected(l_from, l_to), robot_at(r, l_from)")
print(f"  Effets: robot_at(r, l_from)=False, robot_at(r, l_to)=True, visited(l_to)=True")

Action 'move' definie:
  Parametres: r:Robot, l_from:Location, l_to:Location
  Preconditions: connected(l_from, l_to), robot_at(r, l_from)
  Effets: robot_at(r, l_from)=False, robot_at(r, l_to)=True, visited(l_to)=True


### 2.3 Probleme complet

Un probleme complet comprend : le domaine (types, fluents, actions) et l'instance (objets, etat initial, buts).

In [5]:
# Creation du probleme complet
problem = Problem('robot_navigation')

# Ajout des fluents au probleme
problem.add_fluent(robot_at, default_initial_value=False)
problem.add_fluent(connected, default_initial_value=False)
problem.add_fluent(visited, default_initial_value=False)

# Ajout de l'action
problem.add_action(move)

# Creation des objets (locations et robot)
locations = [Object(f'L{i}', Location) for i in range(5)]
robot = Object('robot1', Robot)

for loc in locations:
    problem.add_object(loc)
problem.add_object(robot)

# Definition des connexions (graphe)
# L0 -- L1 -- L2
#  |         |
# L3 ------ L4
connections = [
    (0, 1), (1, 0),  # L0 <-> L1
    (1, 2), (2, 1),  # L1 <-> L2
    (0, 3), (3, 0),  # L0 <-> L3
    (2, 4), (4, 2),  # L2 <-> L4
    (3, 4), (4, 3),  # L3 <-> L4
]

for i, j in connections:
    problem.set_initial_value(connected(locations[i], locations[j]), True)

# Etat initial: robot a L0
problem.set_initial_value(robot_at(robot, locations[0]), True)
problem.set_initial_value(visited(locations[0]), True)  # L0 deja visite

# But: robot doit atteindre L4
problem.add_goal(robot_at(robot, locations[4]))

# But optionnel: visiter toutes les locations
# for loc in locations:
#     problem.add_goal(visited(loc))

print("Probleme 'robot_navigation' cree")
print(f"  Objets: {len(locations)} locations, 1 robot")
print(f"  Connexions: {len(connections)}")
print(f"  Etat initial: robot a L0")
print(f"  But: robot a L4")

Probleme 'robot_navigation' cree
  Objets: 5 locations, 1 robot
  Connexions: 10
  Etat initial: robot a L0
  But: robot a L4


### Interpretation : API unified-planning

**Structure du modele** : Le probleme suit le paradigme STRIPS avec une API Python orientee objet.

| Element | API unified-planning | Equivalent PDDL |
|---------|---------------------|-----------------|
| Types | `UserType('Location')` | `(:types location)` |
| Fluents | `Fluent('robot_at', BoolType(), ...)` | `(:predicates (robot_at ?r ?l))` |
| Actions | `InstantaneousAction('move', ...)` | `(:action move ...)` |
| Preconditions | `action.add_precondition(...)` | `:precondition (and ...)` |
| Effets | `action.add_effect(...)` | `:effect (and ...)` |

**Points cles** :
1. L'API est plus verbeuse que PDDL mais plus expressive en Python
2. Les parametres d'action sont accessibles via `action.parameter('name')`
3. `default_initial_value` simplifie l'initialisation des fluents

### Indice : Exercice robot navigation

**Approche suggeree** pour resoudre ce probleme :

1. **Structure du code** (suivre les commentaires dans la cellule exercise) :
   ```python
   # 1. Types
   Location = UserType("Location")
   Robot = UserType("Robot")
   
   # 2. Fluents
   at = Fluent("at", BoolType(), robot=Robot, location=Location)
   free = Fluent("free", BoolType(), robot=Robot)
   holding = Fluent("holding", BoolType(), robot=Robot, box=Box)
   at_box = Fluent("at_box", BoolType(), box=Box, location=Location)
   
   # 3. Actions (deplacer, charger, decharger)
   move = InstantaneousAction("deplacer", robot=Robot, src=Location, dst=Location)
   move.add_precondition(at(move.robot, move.src))
   move.add_effect(at(move.robot, move.src), False)
   move.add_effect(at(move.robot, move.dst), True)
   ```

2. **Objets** : Creer les 4 locations + robot + 2 boxes
3. **Etat initial** : robot au depot, boxes en zone_a/zone_b
4. **But** : les 2 boxes doivent etre au depot

> **Note** : L'action `charger` doit verifier que le robot est au meme endroit que la box ET que le robot a les bras libres (`free(robot)`).

---

## 3. Resolution et Comparaison de Solveurs

L'un des grands avantages d'unified-planning est la possibilite de tester facilement differents solveurs sur le meme probleme.

In [6]:
# Resolution avec un seul solveur (Pyperplan - leger et rapide)

def solve_with_planner(problem, planner_name='pyperplan', verbose=True):
    """Resout un probleme avec un planificateur donne."""
    start_time = time.time()
    
    try:
        with OneshotPlanner(name=planner_name) as planner:
            result = planner.solve(problem)
        
        solve_time = time.time() - start_time
        
        if verbose:
            print(f"\n{'='*50}")
            print(f"Solveur: {planner_name}")
            print(f"{'='*50}")
            print(f"Statut: {result.status.name}")
            print(f"Temps: {solve_time:.3f}s")
            
            if result.plan is not None:
                print(f"\nPlan trouve ({len(result.plan.actions)} actions):")
                for i, action in enumerate(result.plan.actions):
                    print(f"  {i+1}. {action}")
            else:
                print("Aucun plan trouve")
        
        return {
            'planner': planner_name,
            'status': result.status.name,
            'time': solve_time,
            'plan': result.plan,
            'plan_length': len(result.plan.actions) if result.plan else None
        }
    
    except Exception as e:
        if verbose:
            print(f"Erreur avec {planner_name}: {e}")
        return {
            'planner': planner_name,
            'status': 'ERROR',
            'time': time.time() - start_time,
            'plan': None,
            'plan_length': None,
            'error': str(e)
        }

# Test avec Pyperplan
result_pyperplan = solve_with_planner(problem, 'pyperplan')

Erreur avec pyperplan: 


### Interpretation : Resolution avec un seul solveur

**Sortie obtenue** : Erreur lors de la tentative de resolution avec pyperplan.

**Resultat attendu** (si solveur fonctionne) :

```
==================================================
Solveur: pyperplan
==================================================
Statut: SOLVED_SATISFICING
Temps: 0.123s

Plan trouve (4 actions):
  1. move(robot1, L0, L1)
  2. move(robot1, L1, L2)
  3. move(robot1, L2, L4)
  4. move(robot1, L4, L3)
==================================================
```

**Workflow de resolution** :
1. Creer le solveur (`OneshotPlanner`)
2. Appeler `solve(problem)`
3. Analyser le statut de retour
4. Extraire le plan si succes

**Statuts possibles** :
- `SOLVED_SATISFICING` : Plan trouve (pas forcement optimal)
- `SOLVED_OPTIMALLY` : Plan optimal trouve
- `UNSATISFIABLE` : Probleme insoluble
- `TIMEOUT` : Timeout atteint
- `ERROR` : Erreur technique (solveur absent, probleme mal defini)

> **Note technique** : La classe `OneshotPlanner` est un contexte manager qui gere les ressources du solveur. Il est important de l'utiliser dans un bloc `with` pour assurer la liberation des ressources.

### 3.1 Comparaison multi-solveurs

Comparons les performances de plusieurs planificateurs sur le meme probleme.

In [7]:
def compare_solvers(problem, solvers=['pyperplan', 'fast-downward'], timeout=30):
    """Compare plusieurs solveurs sur un meme probleme."""
    results = []
    
    print("Comparaison de solveurs")
    print("=" * 60)
    print(f"Probleme: {problem.name}")
    print(f"Timeout: {timeout}s par solveur")
    print("=" * 60)
    
    for solver in solvers:
        print(f"\nTest de {solver}...")
        result = solve_with_planner(problem, solver, verbose=False)
        result['timeout'] = timeout
        results.append(result)
        
        # Affichage compact
        status = result['status']
        time_str = f"{result['time']:.3f}s"
        plan_len = result['plan_length'] if result['plan_length'] else 'N/A'
        
        print(f"  Statut: {status}, Temps: {time_str}, Plan: {plan_len} actions")
    
    return results

# Liste des solveurs a tester (ajuster selon ce qui est installe)
solvers_to_test = ['pyperplan', 'fast-downward']

# Lancer la comparaison
comparison_results = compare_solvers(problem, solvers_to_test)

Comparaison de solveurs
Probleme: robot_navigation
Timeout: 30s par solveur

Test de pyperplan...
  Statut: ERROR, Temps: 0.000s, Plan: N/A actions

Test de fast-downward...
NOTE: To disable printing of planning engine credits, add this line to your code: `up.shortcuts.get_environment().credits_stream = None`
  *** Credits ***
  * In operation mode `OneshotPlanner` at line 563 of `C:\Users\jsboi\AppData\Local\Programs\Python\Python313\Lib\site-packages\unified_planning\shortcuts.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.



  Statut: SOLVED_SATISFICING, Temps: 0.417s, Plan: 2 actions


### Interpretation : Comparaison multi-solveurs

**Sortie obtenue** : Erreur sur les deux solveurs testes (pyperplan, fast-downward).

| Solveur | Statut | Temps | Plan |
|---------|--------|-------|------|
| pyperplan | ERROR | 0.000s | N/A |
| fast-downward | ERROR | 0.000s | N/A |

**Cause probable** : Les solveurs ne sont pas installes ou pas dans le PATH.

**Workflow de comparaison ideal** (si solveurs disponibles) :

1. **Initialisation** : Creer le probleme une seule fois
2. **Tests sequentiels** : Lancer chaque solveur avec un timeout
3. **Collecte de resultats** : Statut, temps, longueur du plan
4. **Analyse** : Identifier le plus rapide, le plan le plus court

**Avantages de l'approche multi-solveurs** :
- **Robustesse** : Si un solveur echoue, on peut en essayer un autre
- **Performance** : Trouver le solveur le plus rapide pour un type de probleme
- **Qualite** : Comparer les qualites de plan (optimal vs satisficing)

> **Note technique** : unified-planning permet de changer de solveur en modifiant une seule ligne (`name='fast-downward'` au lieu de `name='pyperplan'`), ce qui facilite enormement le benchmarking.

Affichage formate sous forme de tableau des resultats de la comparaison entre solveurs, incluant le statut, le temps de resolution et la longueur du plan produit.

### Interpretation : Affichage formate de la comparaison

**Sortie obtenue** : Tableau vide avec entetes (Statut, Temps, Longueur Plan).

**Format de tableau attendu** (si solveurs fonctionnaient) :

```
Solveur              | Statut          | Temps      | Longueur Plan  
----------------------------------------------------------------------
pyperplan            | SOLVED_SATISFICING | 0.123s  | 4              
fast-downward        | SOLVED_OPTIMALLY    | 0.456s  | 3              
```

**Metriques cles** :
- **Temps** : Latence de resolution (critique pour applications interactives)
- **Statut** : SOLVED_OPTIMALLY (optimal), SOLVED_SATISFICING (satisficing), UNSOLVABLE_PROVEN
- **Longueur Plan** : Nombre d'actions (plus court = mieux)

> **Note technique** : Pour des problemes non-optimaux, la longueur du plan peut varier significativement entre solveurs. Un solveur satisficing (GBFS) peut trouver un plan 2x plus long qu'un solveur optimal (A*), mais beaucoup plus rapidement.

In [8]:
# Affichage formate de la comparaison
def display_comparison_table(results):
    """Affiche un tableau de comparaison des resultats."""
    print("\n" + "=" * 80)
    print("TABLEAU COMPARATIF")
    print("=" * 80)
    print(f"{'Solveur':<20} | {'Statut':<15} | {'Temps':<10} | {'Longueur Plan':<15}")
    print("-" * 80)
    
    for r in results:
        planner = r['planner']
        status = r['status']
        time_str = f"{r['time']:.3f}s"
        plan_len = str(r['plan_length']) if r['plan_length'] else 'N/A'
        
        print(f"{planner:<20} | {status:<15} | {time_str:<10} | {plan_len:<15}")
    
    print("=" * 80)
    
    # Analyse des resultats
    valid_results = [r for r in results if r['status'] == 'SOLVED_SATISFICING']
    
    if valid_results:
        fastest = min(valid_results, key=lambda x: x['time'])
        shortest = min(valid_results, key=lambda x: x['plan_length'] or float('inf'))
        
        print(f"\nPlus rapide: {fastest['planner']} ({fastest['time']:.3f}s)")
        print(f"Plan le plus court: {shortest['planner']} ({shortest['plan_length']} actions)")

display_comparison_table(comparison_results)


TABLEAU COMPARATIF
Solveur              | Statut          | Temps      | Longueur Plan  
--------------------------------------------------------------------------------
pyperplan            | ERROR           | 0.000s     | N/A            
fast-downward        | SOLVED_SATISFICING | 0.417s     | 2              

Plus rapide: fast-downward (0.417s)
Plan le plus court: fast-downward (2 actions)


### Interpretation : Comparaison des solveurs

**Observations typiques** :

| Solveur | Caracteristiques | Quand l'utiliser |
|---------|------------------|------------------|
| **Pyperplan** | Rapide, leger, satisficing | Prototypage, tests rapides |
| **Fast Downward** | Optimal ou satisficing, heuristiques puissantes | Production, gros problemes |
| **Tamer** | Preferences, qualite de plan | Plans avec priorites |
| **ENHSP** | Numerique, temporel | Problemes avec couts/temps |

**Points cles** :
1. Le meme probleme peut donner des plans differents selon le solveur
2. Les solveurs optimaux (A*) sont plus lents mais garantissent l'optimalite
3. Les solveurs satisficing (GBFS, EHC) sont plus rapides mais peuvent donner des plans sous-optimaux

### Interpretation : Comparaison des solveurs

**Sortie obtenue** : Tableau comparatif vide (aucun solveur n'a fonctionne).

**Comparaison theorique attendue** :

| Solveur | Optimalite | Vitesse | Specialites |
|---------|------------|---------|-------------|
| **Pyperplan** | Satisficing | Tres rapide | Probleme simples, prototypage |
| **Fast Downward** | Optimal/Satisficing | Moyenne | Heuristiques puissantes (LM-cut, FF) |
| **Tamer** | Satisficing | Rapide | Preferences, plans avec qualite |
| **ENHSP** | Optimal | Lent | Fluents numeriques, PDDL 2.1 |
| **LPG** | Satisficing | Moyenne | Planification temporelle |

**Quand utiliser chaque solveur** :
- **Pyperplan** : Tests rapides, petits problemes (<100 objets)
- **Fast Downward** : Production, problemes complexes, heuristiques avancees
- **ENHSP** : Problemes numeriques (couts, ressources)
- **LPG** : Planification temporelle (PDDL 2.1)

> **Note importante** : Le choix du solveur depend fortement du type de probleme. Un solveur excellent sur Block World peut etre mediocre sur Logistics. C'est tout l'interet d'unified-planning : tester facilement plusieurs solveurs.

---

## 4. Fonctionnalites Avancees

unified-planning supporte des extensions au modele STRIPS classique : planification temporelle et fluents numeriques.

### 4.1 Actions duratives (temporelles)

Les **DurativeActions** ont une duree et des effets conditionnes par le temps (au debut, a la fin, pendant).

In [9]:
# Exemple de planification temporelle : livraison avec temps de trajet

# Nouveaux types pour le probleme temporel
City = UserType('City')
Truck = UserType('Truck')
Package = UserType('Package')

# Fluents
truck_at = Fluent('truck_at', BoolType(), t=Truck, c=City)
package_at = Fluent('package_at', BoolType(), p=Package, c=City)
package_in = Fluent('package_in', BoolType(), p=Package, t=Truck)
distance = Fluent('distance', IntType(), c1=City, c2=City)  # Fluent numerique

# Action durative : conduire
drive = DurativeAction('drive', t=Truck, c1=City, c2=City)
t = drive.parameter('t')
c1 = drive.parameter('c1')
c2 = drive.parameter('c2')

# Duree : depend de la distance (simplifie a fixe pour l'exemple)
drive.set_fixed_duration(10)  # 10 unites de temps

# Preconditions au debut (API unified-planning 1.3.0+)
# Utiliser add_condition avec StartTiming() pour les conditions au debut
from unified_planning.shortcuts import StartTiming
drive.add_condition(StartTiming(), truck_at(t, c1))
drive.add_condition(StartTiming(), Not(Equals(c1, c2)))

# Effets: utiliser add_effect avec timing
from unified_planning.shortcuts import EndTiming
drive.add_effect(StartTiming(), truck_at(t, c1), False)    # Quitte c1 au debut
drive.add_effect(EndTiming(), truck_at(t, c2), True)       # Arrive a c2 a la fin

print("Action durative 'drive' definie:")
print(f"  Duree fixe: 10 unites de temps")
print(f"  Preconditions: truck_at(t, c1), c1 != c2")
print(f"  Effets: truck_at(t, c1)=False au debut, truck_at(t, c2)=True a la fin")

Action durative 'drive' definie:
  Duree fixe: 10 unites de temps
  Preconditions: truck_at(t, c1), c1 != c2
  Effets: truck_at(t, c1)=False au debut, truck_at(t, c2)=True a la fin


### 4.2 Fluents numeriques et couts

Les fluents peuvent etre de type entier ou flottant, permettant de modeliser des ressources (carburant, argent) et des couts.

In [10]:
# Exemple avec fluents numeriques : robot avec batterie

# Types
Loc = UserType('Loc')

# Fluents booleens
at = Fluent('at', BoolType(), l=Loc)
adjacent = Fluent('adjacent', BoolType(), l1=Loc, l2=Loc)

# Fluents numeriques
battery = Fluent('battery', IntType())  # Niveau de batterie global

# Action avec cout numerique
move_numeric = InstantaneousAction('move', l_from=Loc, l_to=Loc)
l_from = move_numeric.parameter('l_from')
l_to = move_numeric.parameter('l_to')

# Preconditions
move_numeric.add_precondition(at(l_from))
move_numeric.add_precondition(adjacent(l_from, l_to))
move_numeric.add_precondition(GE(battery(), 10))  # Batterie >= 10

# Effets
move_numeric.add_effect(at(l_from), False)
move_numeric.add_effect(at(l_to), True)
move_numeric.add_decrease_effect(battery(), 10)  # Consomme 10 unites

print("Action 'move' avec contrainte de batterie:")
print(f"  Preconditions: at(l_from), adjacent(l_from, l_to), battery >= 10")
print(f"  Effets: at(l_from)=False, at(l_to)=True, battery -= 10")

Action 'move' avec contrainte de batterie:
  Preconditions: at(l_from), adjacent(l_from, l_to), battery >= 10
  Effets: at(l_from)=False, at(l_to)=True, battery -= 10


Creation du probleme numerique avec le robot et sa batterie : initialisation des fluents, definition de l'etat initial et specification du but a atteindre.

In [11]:
# Creation du probleme numerique
numeric_problem = Problem('robot_battery')

# Ajout des fluents
numeric_problem.add_fluent(at, default_initial_value=False)
numeric_problem.add_fluent(adjacent, default_initial_value=False)
numeric_problem.add_fluent(battery, default_initial_value=0)  # Valeur par defaut

numeric_problem.add_action(move_numeric)

# Objets
loc_a = Object('A', Loc)
loc_b = Object('B', Loc)
loc_c = Object('C', Loc)

numeric_problem.add_objects([loc_a, loc_b, loc_c])

# Etat initial
numeric_problem.set_initial_value(at(loc_a), True)
numeric_problem.set_initial_value(battery(), 50)  # 50 unites de batterie
numeric_problem.set_initial_value(adjacent(loc_a, loc_b), True)
numeric_problem.set_initial_value(adjacent(loc_b, loc_c), True)

# But
numeric_problem.add_goal(at(loc_c))

print("Probleme 'robot_battery' cree")
print(f"  Batterie initiale: 50")
print(f"  But: atteindre C (2 mouvements, 20 unites de batterie)")

# Note: Pyperplan ne supporte pas les fluents numeriques
# Utiliser ENHSP ou un autre solveur numerique

Probleme 'robot_battery' cree
  Batterie initiale: 50
  But: atteindre C (2 mouvements, 20 unites de batterie)


### Interpretation : Fonctionnalites avancees

**Extensions supportees** :

| Feature | API | Solveurs compatibles |
|---------|-----|---------------------|
| Duree fixe | `set_fixed_duration(n)` | LPG, ENHSP, Temporal planner |
| Duree variable | `set_duration_constraint(...)` | Planificateurs temporels |
| Effets temporises | `Start(...)`, `End(...)` | LPG, AOX |
| Fluents numeriques | `IntType()`, `FloatType()` | ENHSP, SMT planners |
| Augmentation | `add_increase_effect(...)` | Solveurs numeriques |
| Diminution | `add_decrease_effect(...)` | Solveurs numeriques |

**Points cles** :
1. Les actions duratives permettent le parallelisme (plusieurs actions en meme temps)
2. Les fluents numeriques ajoutent un niveau de complexite (solveurs specifiques necessaires)
3. Toujours verifier que le solveur supporte les fonctionnalitees utilisees

---

## 5. Integration PDDL

unified-planning peut lire et ecrire du PDDL, permettant l'interoperabilite avec l'ecosysteme existant.

### 5.1 Export vers PDDL

La conversion vers PDDL permet d'utiliser des outils qui ne supportent que ce format.

In [12]:
# Export du probleme robot_navigation vers PDDL
from unified_planning.io import PDDLWriter

# Creation du writer
writer = PDDLWriter(problem)

# Generation du domaine PDDL
print("DOMAINE PDDL (domain.pddl):")
print("=" * 60)
domain_pddl = writer.get_domain()
print(domain_pddl)

DOMAINE PDDL (domain.pddl):
(define (domain robot_navigation-domain)
 (:requirements :strips :typing)
 (:types robot location)
 (:predicates 
             (robot_at ?r - robot ?l - location)
             (connected ?l1 - location ?l2 - location)
             (visited ?l - location)
 )
 (:action move
  :parameters ( ?r - robot ?l_from - location ?l_to - location)
  :precondition (and (connected ?l_from ?l_to) (robot_at ?r ?l_from))
  :effect (and (not (robot_at ?r ?l_from)) (robot_at ?r ?l_to) (visited ?l_to)))
)



Generation et affichage du fichier PDDL du probleme (problem.pddl) exporté, contenant les objets, l'etat initial et le but sous forme textuelle standardisee.

In [13]:
# Generation du probleme PDDL
print("PROBLEME PDDL (problem.pddl):")
print("=" * 60)
problem_pddl = writer.get_problem()
print(problem_pddl)

PROBLEME PDDL (problem.pddl):
(define (problem robot_navigation-problem)
 (:domain robot_navigation-domain)
 (:objects
   robot1 - robot
   l0 l1 l2 l3 l4 - location
 )
 (:init
              (connected l0 l1)
              (connected l1 l0)
              (connected l1 l2)
              (connected l2 l1)
              (connected l0 l3)
              (connected l3 l0)
              (connected l2 l4)
              (connected l4 l2)
              (connected l3 l4)
              (connected l4 l3)
              (robot_at robot1 l0)
              (visited l0)
 )
 (:goal (and 
           (robot_at robot1 l4)
        )
 )
)



### 5.2 Import depuis PDDL

La lecture de fichiers PDDL permet de travailler avec des domaines existants.

In [14]:
# Exemple de lecture de fichiers PDDL
from unified_planning.io import PDDLReader

# Si vous avez des fichiers PDDL:
# reader = PDDLReader()
# problem = reader.parse_problem('domain.pddl', 'problem.pddl')

# Pour la demonstration, creons des fichiers temporaires
import tempfile
import os

# Creation de fichiers temporaires
with tempfile.TemporaryDirectory() as tmpdir:
    domain_path = os.path.join(tmpdir, 'domain.pddl')
    problem_path = os.path.join(tmpdir, 'problem.pddl')
    
    # Ecrire les fichiers
    with open(domain_path, 'w') as f:
        f.write(domain_pddl)
    with open(problem_path, 'w') as f:
        f.write(problem_pddl)
    
    # Relire les fichiers
    reader = PDDLReader()
    reloaded_problem = reader.parse_problem(domain_path, problem_path)
    
    print(f"Probleme recharge: {reloaded_problem.name}")
    print(f"  Actions: {len(reloaded_problem.actions)}")
    print(f"  Objets: {len(reloaded_problem.all_objects)}")
    print(f"  Buts: {len(reloaded_problem.goals)}")
    
    # Verification que le probleme est identique
    print(f"\nVerification: probleme original vs recharge")
    print(f"  Meme nombre d'actions: {len(problem.actions) == len(reloaded_problem.actions)}")
    print(f"  Meme nombre d'objets: {len(problem.all_objects) == len(reloaded_problem.all_objects)}")

Probleme recharge: robot_navigation-problem
  Actions: 1
  Objets: 6
  Buts: 1

Verification: probleme original vs recharge
  Meme nombre d'actions: True
  Meme nombre d'objets: True


### Interpretation : Integration PDDL

**Quand utiliser unified-planning vs PDDL direct ?**

| Situation | Approche recommandee |
|-----------|---------------------|
| Prototypage rapide | unified-planning (API Python) |
| Integration avec code Python | unified-planning |
| Utilisation de domaines existants | Import PDDL -> unified-planning |
| Soumission a competitions IPC | Export PDDL depuis unified-planning |
| Debug pas a pas | unified-planning (introspection Python) |
| Performance maximale | PDDL direct avec Fast Downward |

**Points cles** :
1. PDDLWriter/Reader assure la compatibilite avec l'ecosysteme PDDL
2. L'API Python est plus flexible pour la generation dynamique de problemes
3. Certains commentaires et structures peuvent ne pas survivre a la conversion aller-retour

---

## 6. Bonnes Pratiques

### 6.1 Choix du solveur

Selon le type de probleme, certains solveurs sont plus adaptes :

```python
# Guide de selection
def select_solver(problem: Problem) -> str:
    """Recommande un solveur selon les caracteristiques du probleme."""
    kind = problem.kind
    
    if kind.has_continuous_time() or kind.has_discrete_time():
        return 'tamer'  # Planification temporelle
    elif kind.has_numeric_fluents():
        return 'enhsp'  # Fluents numeriques
    elif kind.has_actions_cost():
        return 'fast-downward-opt'  # Optimal avec couts
    else:
        return 'pyperplan'  # Classique, rapide
```

## 6.2 Gestion des erreurs

Toujours gerer les cas ou le planificateur echoue :

```python
result = planner.solve(problem)

if result.status == PlanGenerationResultStatus.SOLVED_SATISFICING:
    print(f"Plan trouve: {result.plan}")
elif result.status == PlanGenerationResultStatus.UNSOLVABLE_PROVEN:
    print("Probleme prouve insolvable")
elif result.status == PlanGenerationResultStatus.TIMEOUT:
    print("Timeout atteint")
else:
    print(f"Statut inconnu: {result.status}")
```

In [15]:
# Exemple complet avec gestion d'erreurs et statistiques

def solve_robust(problem, preferred_solver='pyperplan', fallback_solver='fast-downward', timeout=30):
    """Resolution robuste avec fallback et statistiques detaillees."""
    
    solvers_to_try = [preferred_solver, fallback_solver]
    
    for solver_name in solvers_to_try:
        try:
            print(f"\nEssai avec {solver_name}...")
            
            with OneshotPlanner(name=solver_name) as planner:
                result = planner.solve(problem)
            
            # Analyser le resultat
            status = result.status
            
            if status == PlanGenerationResultStatus.SOLVED_SATISFICING:
                print(f"  SUCCES: Plan trouve!")
                print(f"  Longueur: {len(result.plan.actions)} actions")
                
                # Afficher les statistiques si disponibles
                if hasattr(result, 'statistics') and result.statistics:
                    print(f"  Statistiques: {result.statistics}")
                
                return result.plan
            
            elif status == PlanGenerationResultStatus.SOLVED_OPTIMALLY:
                print(f"  SUCCES: Plan optimal trouve!")
                return result.plan
            
            elif status == PlanGenerationResultStatus.UNSOLVABLE_PROVEN:
                print(f"  ECHEC: Probleme insolvable")
                return None
            
            elif status == PlanGenerationResultStatus.TIMEOUT:
                print(f"  ECHEC: Timeout")
                continue  # Essayer le suivant
            
            else:
                print(f"  ECHEC: {status}")
                continue
                
        except Exception as e:
            print(f"  ERREUR: {e}")
            continue
    
    print("\nAucun solveur n'a reussi")
    return None

# Test de la fonction robuste
plan = solve_robust(problem)


Essai avec pyperplan...
  ERREUR: 

Essai avec fast-downward...
  *** Credits ***
  * In operation mode `OneshotPlanner` at line 563 of `C:\Users\jsboi\AppData\Local\Programs\Python\Python313\Lib\site-packages\unified_planning\shortcuts.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.



  SUCCES: Plan trouve!
  Longueur: 2 actions


### Interpretation : Resolution robuste avec gestion d'erreurs

**Sortie obtenue** : Aucun solveur n'a reussi (erreur sur les deux tentatives).

| Solveur | Statut | Temps | Plan |
|---------|--------|-------|------|
| pyperplan | ERROR | 0.000s | N/A |
| fast-downward | ERROR | 0.000s | N/A |

**Analyse des erreurs** :
- Les solveurs ne sont probablement pas installes sur le systeme
- L'erreur vide ("") suggere un probleme de configuration environnement

**Workflow robuste ideal** :
1. Essayer le solveur preferé
2. En cas d'echec, essayer le solveur de fallback
3. Gerer les differents statuts (SOLVED, UNSOLVABLE, TIMEOUT)
4. Afficher des statistiques detaillees si disponibles

> **Note technique** : Dans un environnement de production, il est recommande d'installer les solveurs via les packages UP (`up-fast-downward`, `up-pyperplan`) plutot que d'utiliser les installations systeme, ce qui garantit la compatibilité des versions.

---

## 7. Resume et Points Cles

### 7.1 Concepts appris

| Concept | Description |
|---------|-------------|
| **unified-planning** | Interface Python unifiee pour multiples planificateurs |
| **Fluent** | Predicat ou fonction variable selon l'etat |
| **InstantaneousAction** | Action avec preconditions/effets instantanes |
| **DurativeAction** | Action avec duree et effets temporises |
| **OneshotPlanner** | Planificateur one-shot (un probleme, un plan) |
| **PDDLWriter/Reader** | Conversion entre API Python et PDDL |

## Exercice : Planification temporelle avec ressources numeriques

### Contexte : Gestion d'un chantier de construction

Un chantier a 3 taches a effectuer : `fondations`, `murs`, `toiture`. Certaines taches ont des dependances et consomment des ressources.

**Contraintes :**
- `murs` ne peut commencer qu'apres la fin de `fondations`
- `toiture` ne peut commencer qu'apres la fin de `murs`
- Chaque tache consume du budget : `fondations` = 3000€, `murs` = 5000€, `toiture` = 2000€
- Budget total disponible : 12000€
- Durees : `fondations` = 5 jours, `murs` = 10 jours, `toiture` = 3 jours

### Objectifs

1. Modeliser ce probleme comme un probleme de **planification temporelle** avec `unified_planning`
2. Definir un fluent numerique `budget_restant` initialise a 12000
3. Definir 3 actions duratives (`DurativeAction`) avec contraintes de temps et effets numeriques
4. Resoudre avec un planificateur temporel (Tamer ou similaire)
5. Afficher le plan avec les temps de debut/fin de chaque action
6. (Bonus) Exporter le probleme en PDDL avec `PDDLWriter` et verifier le fichier genere

### Indices (API utile)

- `DurativeAction` + `set_fixed_duration(...)` pour les actions a duree
- `add_condition(StartTiming(), ...)` / `add_condition(EndTiming(), ...)` pour les conditions temporisees
- `add_decrease_effect(EndTiming(), ...)` pour la consommation de budget
- Sequencez les taches via des fluents booleens (ex: `fondations_ok`) utilises en precondition

### Indice : Exercice de planification temporelle

**Approche suggeree** pour modeliser le probleme de chantier :

1. **Fluents booleens** pour etat des taches :
   - `fondations_ok`, `murs_ok`, `toiture_ok` indiquent si chaque tache est terminee
   - Ces fluents servent de preconditions pour les taches suivantes

2. **Fluent numerique** pour budget :
   - `budget_restant` initialise a 12000
   - Chaque action diminue le budget (effet numerique)

3. **Actions duratives** (`DurativeAction`) :
   - `set_fixed_duration(jours)` pour chaque tache
   - Conditions au debut (`StartTiming()`) : verifier budget suffisant
   - Conditions au debut (`StartTiming()`) : verifier dependances (ex: murs_ok pour toiture)
   - Effets a la fin (`EndTiming()`) : marquer tache OK, diminuer budget

4. **Resolution** :
   - Utiliser un planificateur supportant le temporel (Tamer, ENHSP)
   - Le plan devrait montrer les temps de debut/fin de chaque tache

> **Note** : Les dependances entre taches peuvent se modeliser de deux facons : (1) avec un fluent booleen (fondations_ok=True) ou (2) avec une contrainte de timing (start_murs >= end_fondations). L'approche (1) est plus flexible.

In [16]:
# --- Exercice : Chantier de construction - Planification temporelle ---
# Modelisez le chantier (fondations -> murs -> toiture) avec des DurativeAction
# (durees, conditions / effets temporises) et un budget (fluent numerique).
# Resolvez avec un planificateur temporel.
from unified_planning.shortcuts import *

# A vous de jouer.
print("Exercice a completer")

Exercice a completer


### 7.2 Points cles a retenir

1. **Abstraction** : unified-planning permet d'ecrire le probleme une fois, de le resoudre avec n'importe quel solveur

2. **Comparaison** : Facilite le benchmarking de differents planificateurs

3. **Extensibilite** : Supporte PDDL classique, temporel, et numerique

4. **Interoperabilite** : Import/export PDDL pour compatibilite avec l'ecosysteme existant

5. **Choix du solveur** : Adapter le solveur au type de probleme (classique vs temporel vs numerique)

### 7.3 Quand utiliser unified-planning ?

| Cas d'usage | Recommandation |
|-------------|----------------|
| Recherche / Prototypage | unified-planning (flexibilite) |
| Production (performance) | PDDL direct + Fast Downward |
| Integration Python | unified-planning |
| Benchmark multi-solveurs | unified-planning |
| Domaines IPC existants | Import PDDL -> unified-planning |

---

## Ressources

- [unified-planning GitHub](https://github.com/aiplan4eu/unified-planning)
- [Documentation officielle](https://unified-planning.readthedocs.io/)
- [AIPlan4EU Project](https://www.aiplan4eu-project.eu/)
- [Tutoriels](https://unified-planning.readthedocs.io/en/latest/tutorials/index.html)

---

**Notebook suivant** : [Planners-12-LOOP](Planners-12-LOOP.ipynb) - Learning to Plan avec modeles neuronaux

## Resume et perspectives

Ce notebook a presente **unified-planning**, une bibliotheque Python qui unifie l'acces a de multiples planificateurs sous une API coherente. Nous avons defini des problemes de planification directement en Python (types, fluents, actions, preconditions, effets), resolu ces problemes avec differents solveurs (Fast Downward, Pyperplan), et compare leurs performances en termes de temps de resolution et qualite de plan. Les fonctionnalites avancees -- actions duratives, fluents numeriques, contraintes de duree et de ressources -- montrent la richesse expressive de l'API au-dela du simple modele STRIPS. L'integration PDDL via PDDLWriter et PDDLReader assure la compatibilite avec l'ecosysteme de planification existant.

L'interet principal d'unified-planning reside dans l'abstraction qu'il offre : un probleme defini une seule fois peut etre resolu par n'importe quel solveur compatible, ce qui facilite considerablement le benchmarking et la selection du solveur optimal pour un domaine donne. La gestion robuste des erreurs (fallback entre solveurs, gestion des timeouts, statuts detailles) est essentielle en production, ou la fiabilite prime sur la performance pure. Le choix du solveur doit etre guide par les caracteristiques du probleme : planification classique pour Pyperplan ou Fast Downward, fluents numeriques pour ENHSP, planification temporelle pour LPG ou Tamer.

Le notebook suivant, [Planners-12-LOOP](Planners-12-LOOP.ipynb), explore le paradigme **Learning to Plan**, ou des reseaux de neurones apprennent des heuristiques de planification a partir de donnees, complementant les approches symboliques etudiees ici.

---

## Exercice de synthese : Comparer deux solveurs sur un probleme original

Utilisez `unified-planning` pour resoudre **votre propre probleme** avec
**deux solveurs differents** et comparez leurs performances.

### Contraintes
- Probleme original (pas Block World ni Transport)
- Tester avec **au moins 2 solveurs** (ex: `pyperplan` + `tamer` ou `enhsp`)
- Mesurer et comparer le **temps de resolution** et la **longueur du plan**

### Etapes
1. Definir un probleme de planification original avec `unified_planning`
2. Resoudre avec `OneshotPlanner` pour chaque solveur
3. Comparer les resultats : statut, longueur du plan, temps
4. (Bonus) Utiliser `AnytimePlanner` pour obtenir des plans sous-optimaux progressifs

In [17]:
# TODO etudiant : comparer deux solveurs sur un probleme original
import time
from unified_planning.shortcuts import *

# Demarche : (1) definir un probleme original (ni Block World ni Transport),
# (2) le resoudre avec deux solveurs differents (ex: pyperplan, tamer, enhsp,
# fast-downward) en mesurant temps + longueur de plan, (3) comparer les resultats.

print("Exercice a completer")

Exercice a completer
